In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import config

# Verify data is accessible
try:
    config.assert_data_exists()
    print("✓ Data path:", config.DATA_ROOT)
except FileNotFoundError as e:
    print("✗ Data path error:", e)

# Data loading utility — use this instead of pd.read_csv() directly
# It handles column name cleaning (the CSV has spaces in headers)
def load_csv(path):
    """Load a CSV from config.DATA_RAW or config.DATA_PROCESSED with cleaned column names."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    # Also strip whitespace from all string/object column VALUES
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].str.strip()
    return df

# Example usage:
# df = load_csv(config.DATA_RAW / "Airlines.csv")
# df = load_csv(config.DATA_PROCESSED / "Airlines_enriched.csv")

✓ Data path: /home/hareee234/Dev/sjsu/cmpe188-data


# 04 — Hyperparameter Tuning
**CMPE 188 | Flight Delay Prediction**

Goals:
- Load the enriched dataset (from notebook 02)
- Run GridSearchCV on XGBoost — completes the TODO from `scripts/xgboost_pipeline.py`
- Run RandomizedSearchCV on Random Forest
- 5-fold stratified cross-validation throughout
- Compare tuned models against notebook 03 baselines

In [2]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

DATA_PATH = str(config.DATA_PROCESSED / 'Airlines_enriched.csv')
df = load_csv(DATA_PATH)
df.head()

/tmp/ipykernel_9591/3554984150.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:


,id,Airline,Flight,AirportFrom,AirportTo,DayOfWeek,Time,Length,Delay,from_lat,...,from_elevation_ft,from_avg_temperature,from_avg_precipitation,from_avg_wind_speed,to_lat,to_lon,to_elevation_ft,to_avg_temperature,to_avg_precipitation,to_avg_wind_speed
0,1,CO,269,SFO,IAH,3,15,205,1,37.619806,...,13.0,14.806027,1.611836,8.456712,29.984400,-95.341400,97.0,20.315616,2.408658,11.432877
1,2,US,1558,PHX,CLT,3,15,222,1,33.435302,...,1135.0,22.637808,0.967808,8.398904,35.214001,-80.943100,748.0,16.326301,2.777151,9.520274
2,3,AA,2400,LAX,DFW,3,20,165,1,33.942501,...,125.0,17.383836,0.947096,8.374247,32.896801,-97.038002,607.0,18.216712,2.134110,14.586849
3,4,AA,2466,SFO,DFW,3,20,195,1,37.619806,...,13.0,14.806027,1.611836,8.456712,32.896801,-97.038002,607.0,18.216712,2.134110,14.586849
4,5,AS,108,ANC,SEA,3,30,202,0,61.179004,...,152.0,2.723562,2.639945,6.909315,47.447943,-122.310276,433.0,10.675616,4.002575,8.618356


## 1. Preprocessing + Train/Test Split

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# ── Derived feature functions ─────────────────────────────────────────────
def add_time_features(df):
    df = df.copy()
    hours = df["Time"] / 60
    df["time_bucket"] = pd.cut(
        hours, bins=[0, 6, 12, 18, 24],
        labels=["night", "morning", "afternoon", "evening"], right=False,
    ).astype(str)
    df["is_peak_hour"] = (((hours >= 7) & (hours < 9)) | ((hours >= 17) & (hours < 20))).astype(int)
    return df

def add_route_volume(df):
    df = df.copy()
    route_counts = df.groupby(["AirportFrom", "AirportTo"]).transform("count")["Time"]
    df["route_volume"] = route_counts
    return df

df = add_time_features(df)
df = add_route_volume(df)
print("Added time_bucket, is_peak_hour, route_volume")

target = "Delay"
X = df.drop(columns=target)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Airline delay rate (train-only, no leakage) ───────────────────────────
airline_delay_rate = X_train.join(y_train).groupby("Airline")[target].mean()
global_rate = y_train.mean()
X_train = X_train.copy()
X_test = X_test.copy()
X_train["airline_delay_rate"] = X_train["Airline"].map(airline_delay_rate).fillna(global_rate)
X_test["airline_delay_rate"] = X_test["Airline"].map(airline_delay_rate).fillna(global_rate)

# ── Route delay rate (train-only, no leakage) ─────────────────────────────
route_delay_rate_map = X_train.join(y_train).groupby(["AirportFrom", "AirportTo"])[target].mean()
X_train["route_delay_rate"] = X_train.set_index(["AirportFrom", "AirportTo"]).index.map(route_delay_rate_map).fillna(global_rate).values
X_test["route_delay_rate"] = X_test.set_index(["AirportFrom", "AirportTo"]).index.map(route_delay_rate_map).fillna(global_rate).values

# ── Flight sequence delay rate (propagation proxy, train-only) ────────────
_flight_col = load_csv(DATA_PATH)["Flight"]
X_train["_flight"] = _flight_col.loc[X_train.index].values
X_test["_flight"] = _flight_col.loc[X_test.index].values
flight_seq_rate_map = X_train.join(y_train).groupby(["Airline", "_flight", "DayOfWeek"])[target].mean()
X_train["flight_seq_delay_rate"] = X_train.set_index(["Airline", "_flight", "DayOfWeek"]).index.map(flight_seq_rate_map).fillna(global_rate).values
X_test["flight_seq_delay_rate"] = X_test.set_index(["Airline", "_flight", "DayOfWeek"]).index.map(flight_seq_rate_map).fillna(global_rate).values
X_train = X_train.drop(columns=["_flight"])
X_test = X_test.drop(columns=["_flight"])

# ── Column lists ──────────────────────────────────────────────────────────
categorical_cols = ["Airline", "AirportFrom", "AirportTo", "time_bucket"]
numeric_cols = [
    "DayOfWeek", "Time", "Length", "is_peak_hour", "route_volume",
    "airline_delay_rate", "route_delay_rate", "flight_seq_delay_rate",
    # Enriched geographic + weather features
    "from_lat", "from_lon", "from_elevation_ft",
    "from_avg_temperature", "from_avg_precipitation", "from_avg_wind_speed",
    "to_lat", "to_lon", "to_elevation_ft",
    "to_avg_temperature", "to_avg_precipitation", "to_avg_wind_speed",
]

# Keep only columns that exist
categorical_cols = [col for col in categorical_cols if col in X_train.columns]
numeric_cols = [col for col in numeric_cols if col in X_train.columns]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", MinMaxScaler(), numeric_cols),
    ]
)

selector = SelectKBest(score_func=chi2, k=50)

print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Added time_bucket, is_peak_hour, route_volume


/tmp/ipykernel_9591/3554984150.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:


Categorical (4): ['Airline', 'AirportFrom', 'AirportTo', 'time_bucket']
Numeric (20): ['DayOfWeek', 'Time', 'Length', 'is_peak_hour', 'route_volume', 'airline_delay_rate', 'route_delay_rate', 'flight_seq_delay_rate', 'from_lat', 'from_lon', 'from_elevation_ft', 'from_avg_temperature', 'from_avg_precipitation', 'from_avg_wind_speed', 'to_lat', 'to_lon', 'to_elevation_ft', 'to_avg_temperature', 'to_avg_precipitation', 'to_avg_wind_speed']
Train: (431506, 26), Test: (107877, 26)


## 2. GridSearchCV — XGBoost

In [4]:
# --- GPU detection for XGBoost ---
import os
HAS_GPU = os.environ.get("CUDA_VISIBLE_DEVICES", "") != ""
if not HAS_GPU:
    try:
        import subprocess
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        HAS_GPU = result.returncode == 0
    except Exception:
        HAS_GPU = False

if HAS_GPU:
    xgb_kwargs = dict(device="cuda", eval_metric="logloss", random_state=42)
    grid_n_jobs = 1          # GPU: one job at a time to avoid OOM
    print("Using GPU for XGBoost (device=cuda)")
else:
    xgb_kwargs = dict(eval_metric="logloss", random_state=42)
    grid_n_jobs = -1         # CPU: use all cores
    print("No GPU detected — using CPU for XGBoost")

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("selector", selector),
    ("classifier", XGBClassifier(**xgb_kwargs)),
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.01, 0.1, 0.2],
    "classifier__subsample": [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=grid_n_jobs,
    verbose=1,
)

import time as _time
_t0 = _time.time()
xgb_grid.fit(X_train, y_train)
print(f"Wall-clock: {_time.time() - _t0:.1f}s")

print(f"Best Parameters: {xgb_grid.best_params_}")
print(f"Best ROC-AUC (CV): {xgb_grid.best_score_:.4f}")

Using GPU for XGBoost (device=cuda)
Fitting 5 folds for each of 54 candidates, totalling 270 fits


/home/hareee234/miniconda3/envs/torch5070/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [10:41:10] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Wall-clock: 449.8s
Best Parameters: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 300, 'classifier__subsample': 0.8}
Best ROC-AUC (CV): 0.8558


## 3. RandomizedSearchCV — Random Forest

In [5]:
# sklearn's RandomForestClassifier is CPU-only (no GPU support).
# GPU options: cuML RandomForestClassifier (RAPIDS) or skip RF tuning.
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("selector", selector),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

param_distributions = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [5, 10, 15, 20, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__max_features": ["sqrt", "log2", 0.5],
}

rf_random = RandomizedSearchCV(
    rf_pipeline,
    param_distributions,
    n_iter=20,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

_t0 = _time.time()
rf_random.fit(X_train, y_train)
print(f"Wall-clock: {_time.time() - _t0:.1f}s")

print(f"Best Parameters: {rf_random.best_params_}")
print(f"Best ROC-AUC (CV): {rf_random.best_score_:.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


/home/hareee234/miniconda3/envs/torch5070/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Wall-clock: 815.8s
Best Parameters: {'classifier__n_estimators': 100, 'classifier__min_samples_split': 10, 'classifier__max_features': 0.5, 'classifier__max_depth': 10}
Best ROC-AUC (CV): 0.8537


## 4. Baseline vs Tuned Comparison Table

In [6]:
# Evaluate tuned models on test set
xgb_pred = xgb_grid.predict(X_test)
xgb_proba = xgb_grid.predict_proba(X_test)[:, 1]
xgb_tuned_acc = accuracy_score(y_test, xgb_pred)
xgb_tuned_auc = roc_auc_score(y_test, xgb_proba)

rf_pred = rf_random.predict(X_test)
rf_proba = rf_random.predict_proba(X_test)[:, 1]
rf_tuned_acc = accuracy_score(y_test, rf_pred)
rf_tuned_auc = roc_auc_score(y_test, rf_proba)

results = pd.DataFrame({
    "Model": ["XGBoost baseline", "XGBoost tuned", "RF baseline", "RF tuned"],
    "Features": ["enriched + derived"] * 4,
    "ROC-AUC": [0.6895, xgb_tuned_auc, 0.6848, rf_tuned_auc],
    "Accuracy": [0.6438, xgb_tuned_acc, 0.6384, rf_tuned_acc],
})
print(results.to_string(index=False))

           Model           Features  ROC-AUC  Accuracy
XGBoost baseline enriched + derived 0.689500  0.643800
   XGBoost tuned enriched + derived 0.645183  0.620114
     RF baseline enriched + derived 0.684800  0.638400
        RF tuned enriched + derived 0.647832  0.619400
